In [2]:
import os

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torch.utils.data as data
import torchvision.transforms as transforms
import torch.nn.functional as F
from pydicom import Dataset
from PIL import Image
from tqdm import tqdm

from src.utils.dataloader import BreastMNISTLoader, BCDRLoader, BUSBRALoader, INbreastLoader, BUSUCLMLoader, BreastLesionsUSGLoader
from src.models.main.models import (
    QuantumModel,
    ExtentedQuantumModel,
)
from src.utils.training import (
    training_loop,
    test_model,
)

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve, auc

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"Using: {torch.cuda.get_device_name(0)}")
    print(f"CUDA: {torch.version.cuda}")
else:
    print("CUDA is not available. Using CPU.")

Using: NVIDIA GeForce RTX 4070 Ti SUPER
CUDA: 12.4


In [4]:
# loader = BreastMNISTLoader(
#             batch_size=16, 
#             image_size=28,
#         )

# train_loader, test_loader, val_loader = loader.get_dataloaders()
# dataloader = {"train": train_loader, "val": val_loader, "test": test_loader}

In [5]:
# loader = BreastLesionsUSGLoader(
#     batch_size=16,
#     image_size=224,
#     root_dir='src/data/BreastLesionsUSG/png'
# )

# train_loader, test_loader, val_loader = loader.get_dataloaders()

In [6]:
loader = BUSUCLMLoader(
    batch_size=16,
    image_size=28,
    root_dir='src/data/BUS-UCLM/png'
)

train_loader, test_loader, val_loader = loader.get_dataloaders()

Loaded 165 images from src/data/BUS-UCLM/png/train/0
Loaded 45 images from src/data/BUS-UCLM/png/train/1
Loaded 2 images from src/data/BUS-UCLM/png/val/0
Loaded 26 images from src/data/BUS-UCLM/png/val/1
Loaded 7 images from src/data/BUS-UCLM/png/test/0
Loaded 19 images from src/data/BUS-UCLM/png/test/1


In [7]:
# inbreast_loader = INbreastLoader(
#     batch_size=16,
#     image_size=28,
#     root_dir='src/data/INbreast/png'
# )

# train_loader, val_loader, test_loader = inbreast_loader.get_dataloaders()
# inbreast_dataloader = {"train": train_loader, "val": val_loader, "test": test_loader}

In [8]:
# bcdr_loader = BCDRLoader(
#     batch_size=16,
#     image_size=28,
#     root_dir='src/data/BCDR/png'
# )

# train_loader, test_loader, val_loader = bcdr_loader.get_dataloaders()
# bcdr_dataloader = {"train": train_loader, "val": val_loader, "test": test_loader}

In [9]:
n_qubits = 4
n_layers = 1
rand_params = np.random.uniform(high=2 * np.pi, size=(n_layers, n_qubits))
num_classes = 2
patch_size = 2 
output_size = (14, 14)

In [10]:
model_basic = QuantumModel(
    n_qubits=n_qubits,
    n_layers=n_layers,
    rand_params=rand_params,
    output_size=output_size,
    patch_size=patch_size,
    num_classes=num_classes
).to(device)

print(f"QuantumModel created with {sum(p.numel() for p in model_basic.parameters())} parameters")

QuantumModel created with 0 parameters


In [11]:
model_trainable = ExtentedQuantumModel(
    n_qubits=n_qubits,
    n_layers=n_layers,
    num_classes=num_classes,
    patch_size=patch_size,
    mode="2d",
    stride=None,
    use_parallel=True
).to(device)

print(f"ExtentedQuantumModel created")

Quantum layer has 4 trainable parameters
Requires grad: True
ExtentedQuantumModel created


## Select Model for Training

Choose which model to use by uncommenting one of the lines below:

In [12]:
# Select one model for training:
# model = model_basic  # Non-trainable quantum without encoder
model = model_trainable  # Trainable quantum without encoder (3 FC layers)

print(f"\nSelected model: {model.__class__.__name__}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters())}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")


Selected model: ExtentedQuantumModel
Total parameters: 4
Trainable parameters: 4


## Training Configuration

In [20]:
# Training hyperparameters
num_epochs = 20
learning_rate = 3e-1
save_dir = "results/models/no_encoder/busuclm_trainable"
os.makedirs(save_dir, exist_ok=True)

# Initialize FC layer with dummy forward pass (for non-trainable models)
if isinstance(model, (QuantumModel)):
    with torch.no_grad():
        dummy_input = torch.randn(1, 1, 28, 28).to(device)
        _ = model(dummy_input)
    print("✓ Model FC layer initialized")

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=learning_rate
)

print(f"\nTraining configuration:")
print(f"  Epochs: {num_epochs}")
print(f"  Learning rate: {learning_rate}")
print(f"  Loss function: CrossEntropyLoss")
print(f"  Save directory: {save_dir}")


Training configuration:
  Epochs: 20
  Learning rate: 0.3
  Loss function: CrossEntropyLoss
  Save directory: results/models/no_encoder/busuclm_trainable


In [21]:
history = training_loop(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=num_epochs,
    save_dir=save_dir,
    save_best_only=True,
    save_all_epochs=True,
)

print("\n✓ Training completed!")

Epoch 1/20



Epoch 1 Summary:
  Train - Loss: 17.3430, Acc: 0.5000
  Val   - Loss: 1.6843, Acc: 0.9286, Precision: 0.9286, Recall: 1.0000, F1: 0.9630, AUC: 0.5000
  ✓ New best model saved (val_acc: 0.9286) at results/models/no_encoder/busuclm_trainable/best_model.pth
Checkpoint saved to results/models/no_encoder/busuclm_trainable/checkpoints/model_epoch1.pth
  Checkpoint saved at epoch 1
Epoch 2/20



Epoch 2 Summary:
  Train - Loss: 4.2269, Acc: 0.5095
  Val   - Loss: 12.0580, Acc: 0.0714, Precision: 0.0000, Recall: 0.0000, F1: 0.0000, AUC: 0.8077
Checkpoint saved to results/models/no_encoder/busuclm_trainable/checkpoints/model_epoch2.pth
  Checkpoint saved at epoch 2
Epoch 3/20



Epoch 3 Summary:
  Train - Loss: 3.4404, Acc: 0.5286
  Val   - Loss: 12.4247, Acc: 0.0714, Precision: 0.0000, Recall: 0.0000, F1: 0.0000, AUC: 0.8077
Checkpoint saved to results/models/no_encoder/busuclm_trainable/checkpoints/model_epoch3.pth
  Checkpoint saved at epoch 3
Epoch 4/20



Epoch 4 Summary:
  Train - Loss: 8.6372, Acc: 0.4857
  Val   - Loss: 0.8084, Acc: 0.9286, Precision: 0.9286, Recall: 1.0000, F1: 0.9630, AUC: 0.7308
Checkpoint saved to results/models/no_encoder/busuclm_trainable/checkpoints/model_epoch4.pth
  Checkpoint saved at epoch 4
Epoch 5/20


...


Epoch 16 Summary:
  Train - Loss: 6.6414, Acc: 0.5238
  Val   - Loss: 4.9217, Acc: 0.2500, Precision: 1.0000, Recall: 0.1923, F1: 0.3226, AUC: 0.7500
Checkpoint saved to results/models/no_encoder/busuclm_trainable/checkpoints/model_epoch16.pth
  Checkpoint saved at epoch 16
Epoch 17/20



Epoch 17 Summary:
  Train - Loss: 2.5669, Acc: 0.6143
  Val   - Loss: 0.8486, Acc: 0.6786, Precision: 0.9474, Recall: 0.6923, F1: 0.8000, AUC: 0.7308
Checkpoint saved to results/models/no_encoder/busuclm_trainable/checkpoints/model_epoch17.pth
  Checkpoint saved at epoch 17
Epoch 18/20



Epoch 18 Summary:
  Train - Loss: 2.8625, Acc: 0.5810
  Val   - Loss: 1.4155, Acc: 0.7143, Precision: 1.0000, Recall: 0.6923, F1: 0.8182, AUC: 0.7308
Checkpoint saved to results/models/no_encoder/busuclm_trainable/checkpoints/model_epoch18.pth
  Checkpoint saved at epoch 18
Epoch 19/20



Epoch 19 Summary:
  Train - Loss: 1.4448, Acc: 0.6524
  Val   - Loss: 0.6506, Acc: 0.7500, Precision: 0.9130, Recall: 0.8077, F1: 0.8571, AUC: 0.7115
Checkpoint saved to results/models/no_encoder/busuclm_trainable/checkpoints/model_epoch19.pth
  Checkpoint saved at epoch 19
Epoch 20/20



Epoch 20 Summary:
  Train - Loss: 1.1585, Acc: 0.6905
  Val   - Loss: 2.7239, Acc: 0.5714, Precision: 1.0000, Recall: 0.5385, F1: 0.7000, AUC: 0.7500
Checkpoint saved to results/models/no_encoder/busuclm_trainable/checkpoints/model_epoch20.pth
  Checkpoint saved at epoch 20

✓ Final model saved at results/models/no_encoder/busuclm_trainable/final_model.pth

✓ Training completed!


In [23]:
print("Initializing model layers...")
model.eval() 

best_model_path = os.path.join(save_dir, "checkpoints/model_epoch5.pth")
if os.path.exists(best_model_path):
    checkpoint = torch.load(best_model_path, map_location=device, weights_only=True)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✓ Loaded best model from {best_model_path}")
    print(f"  Epoch: {checkpoint.get('epoch', 'N/A')}")
    print(f"  Val Accuracy: {checkpoint.get('val_acc', 'N/A'):.4f}")
else:
    print("⚠ Best model not found, using current weights")

hyperparams = {
    "epochs": num_epochs,
    "batch_size": 16,
    "learning_rate": learning_rate,
    "n_qubits": n_qubits,
    "n_layers": n_layers,
    "patch_size": patch_size,
    "model": model.__class__.__name__
}

test_loss, test_acc, test_prec, test_rec, test_f1, test_auc = test_model(
    model=model,
    dataloader=test_loader,
    criterion=criterion,
    device=device,
    hyperparams=hyperparams,
    output_file="noencoder.txt",
    extract_features=False,
    compute_detailed_metrics=True
)

print("\n" + "="*50)
print("FINAL TEST RESULTS")
print("="*50)
print(f"Test Loss:      {test_loss:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")
print(f"Test Precision: {test_prec:.4f}")
print(f"Test Recall:    {test_rec:.4f}")
print(f"Test F1 Score:  {test_f1:.4f}")
print(f"Test AUC:       {test_auc:.4f}")
print("="*50)

Initializing model layers...
✓ Loaded best model from results/models/no_encoder/busuclm_trainable/checkpoints/model_epoch5.pth
  Epoch: 4
  Val Accuracy: 0.7857


Testing: 100%|██████████| 2/2 [00:06<00:00,  3.24s/it]

Test Loss: 0.4552
Accuracy: 0.7692, Precision: 0.7826, Recall: 0.9474, F1: 0.8571, AUC: 0.8045
Macro - Precision: 0.7246, Recall: 0.6165, F1: 0.6286
Micro - Precision: 0.7692, Recall: 0.7692, F1: 0.7692

FINAL TEST RESULTS
Test Loss:      0.4552
Test Accuracy:  0.7692
Test Precision: 0.7826
Test Recall:    0.9474
Test F1 Score:  0.8571
Test AUC:       0.8045


In [ ]:
model.eval()
all_labels = []
all_predictions = []
all_probs = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Generating predictions"):
        images = images.to(device)
        labels = labels.squeeze().to(device)
        
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
        
        all_probs.extend(probs)
        all_predictions.extend(outputs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Confusion Matrix
cm = confusion_matrix(all_labels, all_predictions)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

# ROC Curve
fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

print(f"\n✓ Visualizations complete!")